<a href="https://colab.research.google.com/github/LucasAlegre/vote-network/blob/master/vote_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install and import libraries (need to restart the session after install)

In [ ]:
! pip install python-igraph
# Módulo de visualização (Sigma.js + Graphology). No Colab, baixa do repositório.
import os
if not os.path.exists('graph_viz.py'):
    ! wget -q https://raw.githubusercontent.com/LucasAlegre/vote-network/master/graph_viz.py

In [ ]:
import pandas as pd
import urllib.request
import urllib.error
import random
import time
import importlib
import requests
from datetime import date
import numpy as np
import igraph as ig
from igraph import Graph, summary
from itertools import combinations
import os
import graph_viz
importlib.reload(graph_viz)  # relê o módulo se o arquivo mudou (sem reiniciar o kernel)

## Data Retrieval and Processing

In [ ]:
# Linha do tempo alinhada às legislaturas. Cada legislatura começa em 1º de fevereiro e dura 4 anos
# (52ª: fev/2003–jan/2007, ..., 57ª: fev/2023–jan/2027). O "ano" é o ano legislativo, de fevereiro a
# janeiro, para que cada ano caiba inteiro em uma legislatura (as sessões começam em 2 de fevereiro).
# Os dados abertos da Câmara têm votos nominais só a partir de 2003 (antes disso, 7 votações),
# por isso a série começa na 52ª legislatura.
def legislatura_de(data):
    return 52 + (data.year - 2003 - (data.month < 2)) // 4

def inicio_legislatura(n):
    return pd.Timestamp(2003 + 4 * (n - 52), 2, 1)

hoje = pd.Timestamp(date.today())
PRIMEIRA_LEGISLATURA = 52                    # 52ª (2003–2007)
ULTIMA_LEGISLATURA = legislatura_de(hoje)    # a atual
legislaturas = range(PRIMEIRA_LEGISLATURA, ULTIMA_LEGISLATURA + 1)

# {escala: {período: (início, fim)}}, com o fim exclusivo. Cada escala vira um conjunto de linhas do
# tempo (uma por tema e uma "Geral"); na visualização, escolha a escala embaixo, ao lado do tema.
escalas = {
    'Ano': {str(ano): (pd.Timestamp(ano, 2, 1), pd.Timestamp(ano + 1, 2, 1))
            for ano in range(inicio_legislatura(PRIMEIRA_LEGISLATURA).year, hoje.year + (hoje.month >= 2))},
    'Legislatura': {f'{n}ª ({inicio_legislatura(n).year}–{inicio_legislatura(n + 1).year})':
                    (inicio_legislatura(n), inicio_legislatura(n + 1)) for n in legislaturas},
}
# escalas['Mandato presidencial'] = {'Lula 1': (pd.Timestamp(2003, 1, 1), pd.Timestamp(2007, 1, 1)), ...}  # outras escalas
start_date = min(ini for periodos in escalas.values() for ini, _ in periodos.values())
end_date = max(fim for periodos in escalas.values() for _, fim in periodos.values())
pd.DataFrame([(escala, periodo, ini.date(), (fim - pd.Timedelta(days=1)).date())
              for escala, periodos in escalas.items() for periodo, (ini, fim) in periodos.items()],
             columns=['escala', 'período', 'início', 'fim'])

In [ ]:
# Baixa (uma vez, em data/) e lê os CSVs anuais dos dados abertos da Câmara
os.makedirs('data', exist_ok=True)
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 5.1; en-US; rv:1.9.0.7) Gecko/2009021910 Firefox/3.0.7'}

def download(kind, year, tries=5, **read_csv):
    path = f'data/{kind}-{year}.csv'
    if not os.path.isfile(path):
        url = f'https://dadosabertos.camara.leg.br/arquivos/{kind}/csv/{kind}-{year}.csv'
        for attempt in range(tries):  # o servidor da Câmara falha de vez em quando: tenta de novo
            try:
                with urllib.request.urlopen(urllib.request.Request(url, None, HEADERS), timeout=300) as response:
                    data = response.read()
                break
            except (urllib.error.URLError, TimeoutError, ConnectionError) as err:
                if getattr(err, 'code', None) == 404 or attempt == tries - 1:
                    raise
                time.sleep(5 * (attempt + 1))
        with open(path + '.part', 'wb') as f:  # só vira o arquivo final se o download terminar
            f.write(data)
        os.replace(path + '.part', path)
    return pd.read_csv(path, sep=';', low_memory=False, **read_csv)

# Anos civis dos arquivos (o ano legislativo de 2025 vai até janeiro de 2026)
years = range(start_date.year, min(end_date - pd.Timedelta(days=1), hoje).year + 1)

# Votos nominais (sem votações simbólicas) dentro dos períodos
colunas = ['idVotacao', 'dataHoraVoto', 'voto', 'deputado_id', 'deputado_uri', 'deputado_nome',
           'deputado_siglaPartido', 'deputado_uriPartido', 'deputado_siglaUf', 'deputado_urlFoto']
votes = pd.concat([download('votacoesVotos', year, usecols=colunas) for year in years], ignore_index=True)
votes = votes[votes['voto'] != 'Simbólico'].copy()
votes['dataHoraVoto'] = pd.to_datetime(votes['dataHoraVoto'])
votes = votes[(votes['dataHoraVoto'] >= start_date) & (votes['dataHoraVoto'] < end_date)]
votes = votes.sort_values('dataHoraVoto', ignore_index=True)
votes

In [ ]:
# Temas de cada votação: votação → proposições afetadas (votacoesProposicoes) → temas (proposicoesTemas).
# Os temas ficam no arquivo do ano de APRESENTAÇÃO da proposição, que costuma ser anterior à votação
# (ex.: um PL de 2015 votado em 2023); por isso baixamos os anos de todas as proposições votadas.
vote_props = pd.concat([download('votacoesProposicoes', year) for year in years], ignore_index=True)
vote_props = vote_props[vote_props['idVotacao'].isin(votes['idVotacao'])]

themes = []
for year in sorted(vote_props['proposicao_ano'].dropna().astype(int).unique()):
    try:
        themes.append(download('proposicoesTemas', year))
    except urllib.error.HTTPError as err:
        print(f'proposicoesTemas-{year}: {err}')
themes = pd.concat(themes, ignore_index=True)

vote_themes = (vote_props[['idVotacao', 'proposicao_uri']]
               .merge(themes[['uriProposicao', 'tema']], left_on='proposicao_uri', right_on='uriProposicao')
               [['idVotacao', 'tema']].drop_duplicates(ignore_index=True))
print(f"{vote_themes['idVotacao'].nunique()} de {votes['idVotacao'].nunique()} votações nominais com tema "
      f"({vote_themes['tema'].nunique()} temas)")

In [ ]:
# Votações por tema e período (uma votação pode ter vários temas)
vote_themes['dataHoraVoto'] = vote_themes['idVotacao'].map(votes.groupby('idVotacao')['dataHoraVoto'].min())
in_period = lambda df, ini, fim: df[(df['dataHoraVoto'] >= ini) & (df['dataHoraVoto'] < fim)]

def contagem(periodos):
    geral = {label: in_period(votes, ini, fim)['idVotacao'].nunique() for label, (ini, fim) in periodos.items()}
    por_tema = pd.DataFrame({label: in_period(vote_themes, ini, fim).groupby('tema')['idVotacao'].nunique()
                             for label, (ini, fim) in periodos.items()})
    return pd.concat([pd.DataFrame([geral], index=['Geral']), por_tema])

votacoes_por_tema = pd.concat({escala: contagem(periodos) for escala, periodos in escalas.items()}, axis=1)
votacoes_por_tema = votacoes_por_tema.fillna(0).astype(int)
ordem = votacoes_por_tema.drop('Geral')['Legislatura'].sum(axis=1).sort_values(ascending=False).index
votacoes_por_tema = votacoes_por_tema.loc[['Geral', *ordem]]
votacoes_por_tema

In [ ]:
# Informações de cada votação, para a lista de votações da visualização
votacoes = pd.concat([download('votacoes', year) for year in years], ignore_index=True)
votacoes = votacoes[votacoes['id'].isin(votes['idVotacao'])].drop_duplicates('id')
# Proposição principal de cada votação: a afetada que não é requerimento (se houver)
principal = (vote_props.assign(requerimento=vote_props['proposicao_siglaTipo'].eq('REQ'))
             .sort_values('requerimento', kind='stable').drop_duplicates('idVotacao').set_index('idVotacao')
             .reindex(votacoes['id']))
votacoes_info = pd.DataFrame({
    'id': votacoes['id'].values,
    'data': pd.to_datetime(votacoes['dataHoraRegistro']).values,
    'orgao': votacoes['siglaOrgao'].values,          # PLEN = Plenário; as demais são comissões
    'descricao': votacoes['descricao'].values,
    'aprovacao': votacoes['aprovacao'].values,       # 1 = aprovada, 0 = rejeitada
    'sim': votacoes['votosSim'].values,
    'nao': votacoes['votosNao'].values,
    'outros': votacoes['votosOutros'].values,
    'titulo': principal['proposicao_titulo'].values, # ex.: "PL 2630/2020"
    'ementa': principal['proposicao_ementa'].values,
    'url': [f'https://www.camara.leg.br/proposicoesWeb/fichadetramitacao?idProposicao={int(i)}' if pd.notna(i) else None
            for i in principal['proposicao_id'].values],
})
votacoes_info['temas'] = votacoes_info['id'].map(vote_themes.groupby('idVotacao')['tema'].agg(lambda t: sorted(set(t))))
votacoes_info.sort_values('data', ascending=False).head()

In [ ]:
# Linhas do tempo: "Geral" (todas as votações) + uma por tema
temas = None             # None = todos os temas; ou, p.ex., ['Saúde', 'Educação']
MIN_VOTACOES = 5         # quadros (tema × período) com menos votações ficam de fora
BACKBONE_FRACTION = 0.3  # fração das arestas mantida pelo filtro de disparidade

In [ ]:
def get_total_expenses(deputy_id, start_date, end_date):
    #expenses = []
    total = 0.0
    years = '&'.join(['ano={}'.format(year) for year in range(start_date.year, end_date.year+1)])
    url = "https://dadosabertos.camara.leg.br/api/v2/deputados/{}/despesas?{}&itens=100000&ordem=ASC&ordenarPor=ano".format(deputy_id, years)
    read_all = False
    while not read_all:
        read_all = True
        page = requests.get(url).json()
        if 'dados' not in page:
            break
        total += sum([e['valorDocumento'] for e in page['dados']])
        #this_expenses = [{'tipo': e['tipoDespesa'], 'valor': e['valorDocumento']} for e in page['dados']]
        #expenses.extend(this_expenses)
        for link in page['links']:
            if link['rel'] == 'next':
                url = link['href']
                read_all = False
                break
    return total

def get_deputy_info(deputy_id):
    url = 'https://dadosabertos.camara.leg.br/api/v2/deputados/{}'.format(deputy_id)
    data = requests.get(url).json()['dados']
    ultimoStatus = data['ultimoStatus']
    today = date.today()
    nascimento = pd.to_datetime(data['dataNascimento'])
    idade = today.year - nascimento.year - ((today.month, today.day) < (nascimento.month, nascimento.day))
    return {'sexo': data['sexo'],
            'escolaridade': data['escolaridade'],
            'idade': idade,
            'e-mail': ultimoStatus['email'],
            'situação': ultimoStatus['situacao'],
            'cidade natal': data['municipioNascimento']}

In [ ]:
#%% Mesmo deputado com nomes diferentes: um nome só por deputado
# (atribuição direta na coluna: `votes[col].loc[...] = ` e `inplace=True` não alteram `votes` no pandas 3)
votes['deputado_nome'] = votes.groupby('deputado_id')['deputado_nome'].transform(lambda nomes: sorted(nomes.unique())[0])
#%% Deputados diferentes com o mesmo nome (ex.: pai e filho): acrescenta o ano do primeiro voto de cada um
homonimo = votes.groupby('deputado_nome')['deputado_id'].transform('nunique') > 1
primeiro_ano = votes.groupby('deputado_id')['dataHoraVoto'].transform('min').dt.strftime('%Y').fillna('')
votes['deputado_nome'] = votes['deputado_nome'].where(~homonimo, votes['deputado_nome'] + ' (' + primeiro_ano + ')')
print('Homônimos:', sorted(votes.loc[homonimo, 'deputado_nome'].unique()))

#%% URI de cada sigla na API (a mais frequente), antes de juntar os nomes antigos, para pegar o logo atual
sigla_uri = votes.dropna(subset=['deputado_siglaPartido']).groupby('deputado_siglaPartido')['deputado_uriPartido'].agg(
    lambda uris: uris.value_counts().index[0])

#%% Partidos que mudaram de nome ficam com o nome atual (a cor do partido se mantém na linha do tempo).
# Fusões e incorporações (ex.: DEM + PSL = UNIÃO) não entram aqui: cada partido fica com o nome da época.
votes['deputado_siglaPartido'] = votes['deputado_siglaPartido'].replace({
    'PMDB': 'MDB', 'PFL': 'DEM', 'PPB': 'PP', 'PR': 'PL', 'PMR': 'REPUBLICANOS', 'PRB': 'REPUBLICANOS',
    'PPS': 'CIDADANIA', 'PTN': 'PODE', 'PTdoB': 'AVANTE', 'PSDC': 'DC', 'PEN': 'PATRI', 'PATRIOTA': 'PATRI',
    'SD': 'SOLIDARIEDADE', 'SDD': 'SOLIDARIEDADE'})

# Logo dos partidos. A API da Câmara não tem o logo dos partidos novos ou renomeados (dá 404),
# então para eles usamos as imagens da Wikipédia / Wikimedia Commons.
LOGOS = {
    'PL': 'https://upload.wikimedia.org/wikipedia/commons/1/12/Partido_Liberal_%28Brazil%29_logo.svg',
    'UNIÃO': 'https://upload.wikimedia.org/wikipedia/commons/7/73/Uni%C3%A3o_Brasil_logo.svg',
    'REPUBLICANOS': 'https://upload.wikimedia.org/wikipedia/pt/0/0d/Republicanos_logo.png',
    'MDB': 'https://logodownload.org/wp-content/uploads/2018/04/mdb-logo-partido.png',
    'SOLIDARIEDADE': 'https://upload.wikimedia.org/wikipedia/commons/f/fe/Logomarca_do_Partido_Solidariedade.png',
    'CIDADANIA': 'https://upload.wikimedia.org/wikipedia/commons/d/d7/Logo_do_Cidadania_23.png',
    'NOVO': 'https://upload.wikimedia.org/wikipedia/commons/b/b5/Novo30_AOC.png',
    'PATRI': 'https://upload.wikimedia.org/wikipedia/commons/f/f9/Logomarca_do_Partido_Patriota.png',
    'PRD': 'https://upload.wikimedia.org/wikipedia/commons/d/dc/Partido_Renova%C3%A7%C3%A3o_Democr%C3%A1tica_logo.svg',
    'MISSÃO': 'https://upload.wikimedia.org/wikipedia/pt/4/47/MISS%C3%83O_logo.png',
    'DC': 'https://upload.wikimedia.org/wikipedia/commons/0/08/Logomarca_Democracia_Crist%C3%A3.png',
    'S.PART.': None,  # sem partido
}

def get_json(url, tries=5):
    """GET na API da Câmara com novas tentativas (ela devolve erro 504 com frequência)."""
    for attempt in range(tries):
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError):
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))

def logo_do_partido(sigla, uri):
    if sigla in LOGOS:
        return LOGOS[sigla]
    try:
        return get_json(uri)['dados']['urlLogo']
    except (requests.RequestException, ValueError, KeyError, TypeError) as err:
        print(f'Sem logo de {sigla}: {err}')
        return None

party_logo = {sigla: logo_do_partido(sigla, sigla_uri.get(sigla))
              for sigla in sorted(votes['deputado_siglaPartido'].dropna().unique())}

In [ ]:
# Cores dos partidos: cada partido tem uma cor própria, tirada do logo sempre que possível.
# Usa a cor dominante do logo; se ela ficar parecida demais com a de outro partido, usa uma variação
# dela (mais clara/escura) ou uma cor secundária do logo. Partidos com mais deputados têm prioridade.
cores_dos_logos = graph_viz.logo_palette(party_logo, cache='data/logo_colors.json')
party_colors = graph_viz.distinct_colors(
    cores_dos_logos,
    weights=votes.groupby('deputado_siglaPartido')['deputado_id'].nunique().to_dict(),
    neutral=['S.PART.'])  # sem partido: cinza
# party_colors['PSOL'] = '#f2d300'  # para fixar a cor de um partido à mão
graph_viz.color_table(party_colors, party_logo)

## Graph Construction

In [ ]:
def generalized_similarity(m, min_eps=0.001, max_iter=1000):
    """ Balázs Kovács, "A generalized model of relational similarity," Social Networks, 32(3), July 2010, pp. 197–211
        Based on: https://github.com/dzinoviev/generalizedsimilarity
    """
    arcs0 = m - m.mean(axis=1)[:, np.newaxis]
    arcs1 = m.T - m.mean(axis=0)[:, np.newaxis]

    eps = min_eps + 1
    N = np.eye(m.shape[1])

    iters = 0
    while (eps > min_eps and iters < max_iter) or np.isnan(N).any():
        M = arcs0.dot(N).dot(arcs0.T)
        m = np.sqrt(M.diagonal())
        M = ((M / (m+1e-8)).T / (m+1e-8)).T

        Np = arcs1.dot(M).dot(arcs1.T)
        n = np.sqrt(Np.diagonal())
        Np = ((Np / (n+1e-8)).T / (n+1e-8)).T
        eps = np.abs(Np - N).max()
        N = Np

        iters += 1
    return M

def pearson_correlation(m):
    arcs0 = m - m.mean(axis=1)[:, np.newaxis]
    arcs1 = m.T - m.mean(axis=0)[:, np.newaxis]
    M = arcs0.dot(arcs0.T)
    m = np.sqrt(M.diagonal())
    M = ((M / m).T / m).T
    return M

def filter_edges(edges_list, num_nodes, threshold=None, density=0.1):
    edges, weights = [], []
    if threshold is not None:
        for e in edges_list:
            if e[1] >= threshold:
                edges.append(e[0])
                weights.append(e[1])
    else:
        count = int(num_nodes * (num_nodes - 1) * density / 2)
        edges_list.sort(reverse=True, key=lambda e: e[1])
        edges_list = edges_list[:count]
        edges = [e[0] for e in edges_list]
        weights = [e[1] for e in edges_list]
    return edges, weights

def disparity_backbone(graph, fraction=None, alpha=None, weight='weight'):
    """Filtro de disparidade: M. Á. Serrano, M. Boguñá, A. Vespignani, "Extracting the multiscale backbone
    of complex weighted networks," PNAS 106(16), 2009.
    Mesmo cálculo de netbone.disparity + netbone.filters.fraction_filter/threshold_filter, direto no igraph.

    Para cada aresta (i, j): p = min sobre as pontas v com grau k_v > 1 de (1 - w_ij / s_v) ** (k_v - 1),
    onde s_v é a força (soma dos pesos) de v. Mantém a fração `fraction` das arestas com menor p
    (desempate: maior peso) ou, se `alpha` for dado, as arestas com p < alpha. Vértices isolados são removidos.
    """
    w = np.asarray(graph.es[weight], dtype=float)
    k = np.asarray(graph.degree())
    s = np.asarray(graph.strength(weights=weight), dtype=float)
    src, dst = np.asarray(graph.get_edgelist()).T
    def alpha_at(v):
        with np.errstate(divide='ignore', invalid='ignore'):
            return np.where(k[v] > 1, (1 - w / s[v]) ** (k[v] - 1), np.nan)
    p = np.fmin(alpha_at(src), alpha_at(dst))  # fmin ignora NaN (ponta com grau 1)

    if alpha is not None:
        keep = np.flatnonzero(p < alpha)
    else:
        order = np.lexsort((-w, np.where(np.isnan(p), np.inf, p)))  # p crescente, depois peso decrescente
        keep = order[:int(np.ceil(fraction * graph.ecount()))]
    graph = graph.copy()
    graph.es['p_value'] = p.tolist()
    return graph.subgraph_edges(sorted(keep.tolist()), delete_vertices=True)

In [ ]:
def vote_matrix(vote_data):
    """Matriz deputados × votações (Sim = 1, Não = -1, demais votos = 0), só com quem votou no recorte."""
    vote_data = vote_data.drop_duplicates(['idVotacao', 'deputado_nome'])  # 1º voto de cada deputado (por data)
    valor = vote_data['voto'].map({'Sim': 1.0, 'Não': -1.0}).fillna(0.0)
    return vote_data.assign(valor=valor).pivot(index='deputado_nome', columns='idVotacao', values='valor').fillna(0.0)

def build_graph(vote_data, fraction=BACKBONE_FRACTION):
    """Grafo de similaridade de votos de um recorte (tema × período); None se houver poucas votações."""
    m = vote_matrix(vote_data)
    if m.shape[1] < MIN_VOTACOES:
        return None
    M = generalized_similarity(m.values)
    i, j = np.triu_indices(len(m), k=1)
    sim = M[i, j]
    pos = sim > 0
    if not pos.any():
        return None
    graph = Graph(n=len(m), edges=np.column_stack([i[pos], j[pos]]).tolist(), directed=False)
    graph.vs['name'] = m.index.tolist()
    graph.es['similarity'] = sim[pos].tolist()
    span = (sim[pos].max() - sim[pos].min()) or 1.0
    graph.es['weight'] = ((sim[pos] - sim[pos].min()) / span).tolist()  # pesos normalizados em [0, 1]

    # Backbone: filtro de disparidade
    graph = disparity_backbone(graph, fraction=fraction, weight='weight')

    # Comunidades (Leiden) e centralidades
    graph.vs['community'] = graph.community_leiden(weights='weight', objective_function='modularity', n_iterations=100).membership
    graph['votações'] = int(m.shape[1])
    graph['ids_votacoes'] = m.columns.tolist()  # votações deste quadro (lista de votações da visualização)
    graph['modularidade'] = round(graph.modularity(graph.vs['community'], weights='weight'), 3)
    graph.vs['betweenness'] = graph.betweenness()
    graph.vs['closeness'] = graph.closeness()

    # Dados de cada deputado no recorte (último voto: partido e UF da época)
    info = vote_data.groupby('deputado_nome').last().reindex(graph.vs['name'])
    graph.vs['Partido'] = info['deputado_siglaPartido'].tolist()
    graph.vs['UF'] = info['deputado_siglaUf'].tolist()
    graph.vs['Foto'] = info['deputado_urlFoto'].tolist()
    graph.vs['URL'] = info['deputado_uri'].tolist()
    graph.vs['Página'] = [u.replace('dadosabertos.camara.leg.br/api/v2/deputados/', 'www.camara.leg.br/deputados/')
                          if isinstance(u, str) else None for u in graph.vs['URL']]
    return graph

In [ ]:
random.seed(42)  # Leiden reprodutível (o igraph usa o `random` do Python)
lista_temas = temas if temas is not None else votacoes_por_tema.index.drop('Geral').tolist()
votacoes_do_tema = {'Geral': None}  # None = todas as votações
for tema in lista_temas:
    votacoes_do_tema[tema] = set(vote_themes.loc[vote_themes['tema'] == tema, 'idVotacao'])

# Um grafo por escala × tema × período (leva alguns minutos: ~7 min para 2003–2026)
timelines = {}  # {escala: {tema: {período: grafo}}}
for escala, periodos in escalas.items():
    quadros = {nome: {} for nome in votacoes_do_tema}
    for label, (ini, fim) in periodos.items():
        print(f'\r{escala}: {label}   ', end='', flush=True)  # progresso
        no_periodo = in_period(votes, ini, fim)
        for nome, ids in votacoes_do_tema.items():
            graph = build_graph(no_periodo if ids is None else no_periodo[no_periodo['idVotacao'].isin(ids)])
            if graph is not None:
                graph['inicio'] = ini.strftime('%Y-%m-%d')                           # datas do quadro
                graph['fim'] = (fim - pd.Timedelta(days=1)).strftime('%Y-%m-%d')   # (fim inclusive)
                quadros[nome][label] = graph
    # Rótulos de comunidade estáveis no tempo ("Comunidade 1" de 2024 é a que mais se sobrepõe à de 2023)
    timelines[escala] = {nome: graph_viz.align_labels(frames, attr='community', new_attr='Comunidade', label='Comunidade {}')
                         for nome, frames in quadros.items() if frames}
    print(f"\r{escala}: {sum(map(len, timelines[escala].values()))} grafos em {len(timelines[escala])} linhas do tempo")

In [ ]:
# Modularidade de cada quadro (quanto maior, mais o plenário se divide em blocos que votam juntos)
pd.concat({escala: pd.DataFrame({nome: {p: g['modularidade'] for p, g in frames.items()}
                                 for nome, frames in linhas.items()}).T
           for escala, linhas in timelines.items()}, axis=1)

## Visualization

In [ ]:
# Gera um HTML autocontido (Sigma.js + Graphology + ForceAtlas2) e exibe aqui no notebook.
# Embaixo fica a linha do tempo: escolha a escala (ano ou legislatura) e o tema, e mude o período
# (ou "▶ Animar"); o layout se ajusta ao vivo.
# O arquivo camara.html abre direto em qualquer navegador e pode ser publicado (ex.: GitHub Pages).
graph_viz.show(timelines,
               path='camara.html',
               title='Câmara dos Deputados',
               subtitle='Similaridade de votos nominais',
               logos=party_logo, logo_attr='Partido',  # logos dos partidos no painel de detalhes
               colors={'Partido': party_colors},  # uma cor por partido (tabela acima)
               votacoes=votacoes_info,  # lista de votações: informações da proposição e voto de cada deputado
               votos=votes[['idVotacao', 'deputado_nome', 'voto']],
               layout_seconds=4,       # segundos de ForceAtlas2 ao abrir
               transition_seconds=3,   # segundos de ForceAtlas2 a cada troca de período ou tema
               play_seconds=4,         # intervalo entre períodos no "▶ Animar"
               # fa2_settings={'gravity': 0.5, 'linLogMode': True},  # parâmetros do ForceAtlas2: ajuste ao vivo em "⚙ Parâmetros" e copie de lá
               height=820)